In [ ]:
# BRATS BASED SEGMENTATION PIPELINE (NO TRAINING / INFERENCE ONLY)
#
# EXAMPLE FOLDER STRUCTURE (TO BE ADJUSTED IN THE CODE AS NEEDED):
# /TCGA/
#   /TCGA-02-0003/
#       /TCGA-02-0003_flair.nii.gz
#       /TCGA-02-0003_t1.nii.gz
#       /TCGA-02-0003_t1c.nii.gz
#       /TCGA-02-0003_t2.nii.gz
#   /TCGA-02-0048/
#       ...
#
# LABELED OUTPUT:
# /TCGA/
#   /TCGA-02-0003/
#       /TCGA-02-0003_tumor_pred.nii.gz
#   ...
#
# LABELS
# TC = Tumor Core
# WT = Whole Tumor
# ET = Enhancing Tumor0
#
# PREREQUISITES
# install monai and then download the brats seg bundle:
# python -m monai.bundle download --name brats_mri_segmentation --bundle_dir ~/monai_bundles

In [ ]:
# SETUP IMPORTS

import os
import numpy as np
import nibabel as nib
import torch
from monai.networks.nets import SegResNet
from monai.inferers import sliding_window_inference

In [ ]:
# SETUP FOLDERS

# Data folder
BASE_DIR = "/Volumes/Pegasus32_R6/PROJECTS/Glioblastoma/data/TCGA"

# Monai bundle root
BUNDLE_ROOT = os.path.expanduser("~/monai_bundles/brats_mri_segmentation")

Using device: cpu


In [ ]:
# SETUP TORCH DEVICE

DEVICE = torch.device("cpu")
#DEVICE = torch.device("cuda")

In [ ]:
# SETUP MODEL

def build_brats_network():
    net = SegResNet(
        blocks_down=(1, 2, 2, 4),
        blocks_up=(1, 1, 1),
        init_filters=16,
        in_channels=4,
        out_channels=3, # TC, WT, ET
        dropout_prob=0.2,
    )
    return net


In [ ]:
# LOAD MODEL

def load_brats_model(bundle_root=BUNDLE_ROOT, device=DEVICE):
    ckpt_path = os.path.join(bundle_root, "models", "model.pt")
    if not os.path.exists(ckpt_path):
        raise FileNotFoundError(f"Checkpoint not found: {ckpt_path}")

    ckpt = torch.load(ckpt_path, map_location="cpu")

    net = build_brats_network()
    net.load_state_dict(ckpt, strict=False)

    net.to(device)
    net.eval()
    return net

In [ ]:
# LOAD SUBJECT

def load_brats_subject(subj_id, base_dir=BASE_DIR):
    subj_dir = os.path.join(base_dir, f"{subj_id}")
    if not os.path.isdir(subj_dir):
        raise FileNotFoundError(subj_dir)

    def load_nifti(path, dtype=np.float32):
        nii = nib.load(path)
        data = nii.get_fdata().astype(dtype)
        return data, nii.affine

    # Mri path names
    t1_path   = os.path.join(subj_dir, f"{subj_id}_t1.nii.gz")
    t1c_path  = os.path.join(subj_dir, f"{subj_id}_t1ce.nii.gz")
    t2_path   = os.path.join(subj_dir, f"{subj_id}_t2.nii.gz")
    flair_path= os.path.join(subj_dir, f"{subj_id}_flair.nii.gz")

    # Load nii files
    t1c, affine = load_nifti(t1c_path)
    t1,  _      = load_nifti(t1_path)
    t2,  _      = load_nifti(t2_path)
    flair, _    = load_nifti(flair_path)

    img_np = np.stack([t1c, t1, t2, flair], axis=0)  
    # img_np -> (4, H, W, D)
    # img_np[0] -> T1c vol
    # img_np[1] -> T1 vol
    # img_np[2] -> T2 vol
    # img_np[3] -> FLAIR

    print(f"[load_brats_subject] {subj_id}: image {img_np.shape}")
    return img_np, affine, subj_dir


In [ ]:
# NON ZERO VOXELS PER CHANNEL INTENSITY NORMALIZATION

def zscore_nonzero_per_channel(img_np: np.ndarray) -> np.ndarray:

    out = img_np.copy()
    C = img_np.shape[0]
    for c in range(C):
        ch = out[c]
        mask = ch != 0
        if not np.any(mask):
            continue
        vals = ch[mask]
        mean = vals.mean()
        std = vals.std()
        if std < 1e-6:
            std = 1.0
        ch[mask] = (ch[mask] - mean) / std
        out[c] = ch
    return out


In [ ]:
# CONVERT THE 3 BRATS SIGMOID OUTPUT CHANNELS (TC, WT, ET)
# INTO A SINGLE MULTICLASS LABEL MAP (0, 1, 2, 4) VOXELWISE

def combine_channels_to_labels(probs_bin: torch.Tensor) -> torch.Tensor:

    tc = probs_bin[:, 0:1]
    wt = probs_bin[:, 1:2]
    et = probs_bin[:, 2:3]

    # Background -> 0
    labels = torch.zeros_like(tc, dtype=torch.uint8)

    # ET -> 4
    labels = torch.where(et > 0, torch.tensor(4, dtype=torch.uint8, device=labels.device), labels)
    # TC (but not ET) -> 1
    labels = torch.where((et == 0) & (tc > 0), torch.tensor(1, dtype=torch.uint8, device=labels.device), labels)
    # WT (but not TC or ET) -> 2
    labels = torch.where((et == 0) & (tc == 0) & (wt > 0),
                         torch.tensor(2, dtype=torch.uint8, device=labels.device),
                         labels)
    return labels


In [ ]:
# RUN INFERENCE

def run_brats_inference_on_brats_subject(
    subj_id: str,
    base_dir: str = BASE_DIR,
    bundle_root: str = BUNDLE_ROOT,
    roi_size=(240, 240, 160),
    sw_batch_size: int = 1,
    thr: float = 0.5,
    device: torch.device = DEVICE,
):
    print(f"Run inference: Subject {subj_id}")

    # 1) Load subject
    img_np, affine, subj_dir = load_brats_subject(subj_id, base_dir)

    # 2) Intensity normalization
    img_np_norm = zscore_nonzero_per_channel(img_np)
    img_t = torch.from_numpy(img_np_norm).unsqueeze(0).float().to(device)

    # 4) Load model
    net = load_brats_model(bundle_root, device)

    # 5) Sliding windows
    with torch.no_grad():
        logits = sliding_window_inference(
            inputs=img_t,
            roi_size=roi_size,
            sw_batch_size=sw_batch_size,
            predictor=net,
            overlap=0.5,
        )

    # 6) Sigmoid + threshold
    probs = torch.sigmoid(logits)
    probs_bin = (probs > thr).float()

    # 7) Merge channels to brats labels
    labels_pred = combine_channels_to_labels(probs_bin)
    labels_pred_np = labels_pred.squeeze(0).squeeze(0).cpu().numpy().astype(np.uint8)

    # 8) Save prediction
    out_path = os.path.join(subj_dir, f"{subj_id}_tumor_pred.nii.gz")
    nib.save(nib.Nifti1Image(labels_pred_np, affine), out_path)
    print("Output saved:", out_path)

    return labels_pred_np, img_np_norm, affine, subj_dir

In [ ]:
# MAIN SCRIPT

# Collect subjects
subject_dirs = sorted(glob.glob(os.path.join(BASE_DIR, "TCGA-*")))
subject_ids = [os.path.basename(d) for d in subject_dirs]
print("Found subjects:", subject_ids)

# Inference
out_paths = []

for subj_id in subject_ids:
    pred_labels, _, _, subj_dir = run_brats_inference_on_brats_subject(subj_id)

print("\nEnd")